# APT Relocation Execution Stats

## 1. Install dependencies

In [0]:
# Install the Cosmos SDK on the cluster, then restart Python so the new package is importable.
%pip install --quiet azure-cosmos

## 2. Connect to Cosmos DB

In [0]:
# Connect to Azure Cosmos DB
from azure.cosmos import CosmosClient

# Pull credentials from Databricks secrets (set scope/keys once via the Databricks CLI / UI)
COSMOS_SCOPE   = "orbis-addressing"
COSMOS_ENDPOINT = "https://opas-merone-prod-db.documents.azure.com:443/" #dbutils.secrets.get(scope=COSMOS_SCOPE, key="cosmos-endpoint")
COSMOS_KEY      = "" #dbutils.secrets.get(scope=COSMOS_SCOPE, key="cosmos-key")

DATABASE_NAME  = "opas-merone-prod-db"
CONTAINER_NAME = "address-point-message"

client    = CosmosClient(COSMOS_ENDPOINT, credential=COSMOS_KEY)
database  = client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

print(f"Connected to {DATABASE_NAME}.{CONTAINER_NAME}")

## 3. SDP IDs

SDP IDs for the USA BFP-based APT relocation DC runs (SEACO-5792). Comments capture the run date and state for traceability.

In [0]:
SDP_IDS = [
    # 06-March USA-MI (APT-CHECK not deployed, location updated, no metadata tags)
    "ADDRESS_LOCATION_CORRECTION-06-03-2026-32943ad0-5a15-4cd3-b319-f9915b4fe4f9",
    # 06-March USA-MI (apt check deployed, re-run)
    "ADDRESS_LOCATION_CORRECTION-06-03-2026-662474ef-96bd-4e64-95ed-078d26213640",
    # 06-March USA-MI BFP
    "ADDRESS_LOCATION_CORRECTION-06-03-2026-8e0a5d6b-bdf5-48dd-8d54-d845d812b0e3",
    # 24-March USA-NM BFP
    "ADDRESS_LOCATION_CORRECTION-24-03-2026-e74b7d3c-ace4-4067-846e-ccf32bdfd990",
    # 24-March USA-WI BFP
    "ADDRESS_LOCATION_CORRECTION-24-03-2026-5367a14f-b837-469e-9784-17dcd57d295d",
    # 25-March USA-MI revert APA location
    "ADDRESS_LOCATION_CORRECTION-25-03-2026-221ad66b-e219-4205-bd29-3c795497f0d0",
    # 25-March USA-MI revert APA metadata tags
    "ADDRESS_FIELD_CORRECTION-25-03-2026-7633074d-de66-488d-a8a4-976942775bbd",
    # 25-March USA-NY APA relocation
    "ADDRESS_LOCATION_CORRECTION-25-03-2026-8047bf04-7176-4262-94f5-0fc3ee004f84",
    # 26-March USA-PA APT relocation
    "ADDRESS_LOCATION_CORRECTION-26-03-2026-39c97ac4-37ef-49e2-8fd7-377cf5822cf1",
    # 28-March USA-OH APT relocation
    "ADDRESS_LOCATION_CORRECTION-28-03-2026-81f05eb3-4080-41d2-af26-07e54aebe98c",
    # 30-March USA-DC
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-935be3f2-3a13-4d22-ab40-fe37fc3e0481",
    # 30-March USA-AK
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-ed87964a-94d2-4247-8862-9d6faaa9c666",
    # 30-March USA-HI
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-d3e9e299-faec-4e7e-973d-aac6ea6c4ebe",
    # 30-March USA-WY
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-427d2dd7-5bf6-4fef-aad2-5863e56ebe19",
    # 30-March USA-ND
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-599e4eb4-d20f-4c8a-b6eb-5495f3ae2408",
    # 30-March USA-DE
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-9bc87039-f948-4e46-89de-c42759ff8a83",
    # 30-March USA-RI
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-1f2314c4-f444-44f7-9ed8-40a0466e1edc",
    # 30-March USA-SD
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-1780fa25-afe6-4242-a167-14497f4c4c62",
    # 30-March USA-VT
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-47b366cc-ff7d-4a65-9b70-2eac41f7cf65",
    # 30-March USA-NV
    "ADDRESS_LOCATION_CORRECTION-30-03-2026-04642bf4-6d67-40ff-bd9f-0a10ab6b5311",
    # 31-March USA-UT
    "ADDRESS_LOCATION_CORRECTION-31-03-2026-235cbc86-520e-45d9-ac4b-e2a8c54300f0",
    # 01-April USA-MT
    "ADDRESS_LOCATION_CORRECTION-01-04-2026-e761df89-dd3d-4b03-80b4-a06ed0f52c84",
    # 02-April USA-ID
    "ADDRESS_LOCATION_CORRECTION-02-04-2026-d21e939c-ee15-474d-a426-597c935d7702",
    # 03-April USA-ME (NOTE: source value has 11-char trailing block, verify before use)
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-a9014fc3-bf5f-4718-b9c1-3a602dadba5",
    # 03-April USA-NM
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-56d89077-d144-4928-b40b-15169bfbffac",
    # 03-April USA-NE
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-e8c459ae-5b2f-4779-9fad-e01e1754a5ed",
    # 03-April USA-WV
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-b53674ba-4a4a-4ae1-a531-52f343789b1f",
    # 03-April USA-NH
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-65cc8ec0-067c-4088-b81d-b62cb29d23a8",
    # 03-April USA-KS
    "ADDRESS_LOCATION_CORRECTION-03-04-2026-f57e9647-7035-4986-84f2-950f70df8c53",
    # 05-April USA-OR
    "ADDRESS_LOCATION_CORRECTION-05-04-2026-582cf6ab-81f1-4244-a073-67c03cf9b28a",
    # 05-April USA-AR
    "ADDRESS_LOCATION_CORRECTION-05-04-2026-4196f1a2-dd89-448c-bcdb-b354a66d4b3c",
    # 05-April USA-CO
    "ADDRESS_LOCATION_CORRECTION-05-04-2026-028ad1b7-d3ea-40e5-a499-4346e473826c",
    # 05-April USA-LA
    "ADDRESS_LOCATION_CORRECTION-05-04-2026-1f469ddb-a18c-4264-9ba3-a4ab5a64f4e2",
    # 05-April USA-IA
    "ADDRESS_LOCATION_CORRECTION-05-04-2026-bd1fe02a-7c51-4044-ac7c-21b6e3608477",
    # 08-April USA-AZ
    "ADDRESS_LOCATION_CORRECTION-08-04-2026-29499262-a649-49cb-8e98-47ffe102581e",
    # 08-April USA-KY
    "ADDRESS_LOCATION_CORRECTION-08-04-2026-a5d82bce-fac1-4ddf-b851-34a92055fd1e",
    # 08-April USA-OK
    "ADDRESS_LOCATION_CORRECTION-08-04-2026-e4887310-9fb6-4527-b25c-67002d31c19a",
    # 09-April USA-WA
    "ADDRESS_LOCATION_CORRECTION-09-04-2026-54bd3184-6487-44ed-beb5-ac4ba679d8b9",
    # 09-April USA-CT
    "ADDRESS_LOCATION_CORRECTION-09-04-2026-3442aa49-e345-432f-93aa-2f64f78dd2e3",
    # 09-April USA-SC
    "ADDRESS_LOCATION_CORRECTION-09-04-2026-5f94941c-c09d-4ba3-8df4-cad49c68c5a1",
    # 10-April USA-MS
    "ADDRESS_LOCATION_CORRECTION-10-04-2026-f830f0f0-2401-4876-83ad-b1c90b5bea97",
    # 10-April USA-AL
    "ADDRESS_LOCATION_CORRECTION-10-04-2026-7babb39f-e888-4247-b149-9caf7ada8e7e",
    # 10-April USA-MD
    "ADDRESS_LOCATION_CORRECTION-10-04-2026-b4993a4b-9979-405c-aa34-e325689906ed",
    # 11-April USA-MA
    "ADDRESS_LOCATION_CORRECTION-11-04-2026-e225012b-5d52-4b72-be50-feac8d9585ad",
    # 11-April USA-TN
    "ADDRESS_LOCATION_CORRECTION-11-04-2026-d13dd98d-cbc1-44b2-a4f1-7ca732bf22d6",
    # 12-April USA-VA
    "ADDRESS_LOCATION_CORRECTION-12-04-2026-982a3f78-2282-462d-ba68-6ef393a4eeb7",
    # 12-April USA-IN
    "ADDRESS_LOCATION_CORRECTION-12-04-2026-cd03e56e-5f35-4039-bcf9-f7c0a9d21217",
    # 15-April USA-MO part1
    "ADDRESS_LOCATION_CORRECTION-15-04-2026-e64ce90f-fada-4749-bbf1-b0f0dafed94b",
    # 15-April USA-MO part2
    "ADDRESS_LOCATION_CORRECTION-15-04-2026-42452fcd-bb8c-4eaa-9ef7-c3a6230182bd",
    # 16-April USA-NJ part1
    "ADDRESS_LOCATION_CORRECTION-16-04-2026-6694acbd-11fd-41b5-a19e-7569bc556c0d",
    # 16-April USA-NJ part2
    "ADDRESS_LOCATION_CORRECTION-16-04-2026-8bc283a2-513f-44bc-8041-3e5d9fc86e6e",
    # 19-April USA-NC
    "ADDRESS_LOCATION_CORRECTION-19-04-2026-bf4f533b-4495-4577-8678-922cd81c0c14",
    # 20-April USA-GA
    "ADDRESS_LOCATION_CORRECTION-20-04-2026-30ef5d03-8370-4306-a98e-f01f6c0232b2",
    # 26-April USA-FL
    "ADDRESS_LOCATION_CORRECTION-26-04-2026-8987ffcf-7605-4ab6-afc2-30d95dd67a6b",
    # 27-April USA-IL
    "ADDRESS_LOCATION_CORRECTION-27-04-2026-3edfbbad-810e-4008-b4c7-eb946900e733",
    # 09-May USA-CA
    "ADDRESS_LOCATION_CORRECTION-09-05-2026-63f7c6a2-d0f2-4523-8c4b-e0954944e475",
    # 11-May USA-TX
    "ADDRESS_LOCATION_CORRECTION-11-05-2026-82b523a4-9fda-4d46-8afd-4aaa5bb553b6",
]

print(f"Total SDP IDs: {len(SDP_IDS)}")

## 4. Stats per SDP ID grouped by status

For each sdpId in `SDP_IDS`, run a Cosmos `GROUP BY` on `c.status` and collect the per-status document counts into a single DataFrame.

In [0]:
import pandas as pd

# Cosmos Python SDK rejects "SELECT status, COUNT(1) ... GROUP BY status" (non-VALUE aggregate).
# Workaround: discover distinct statuses for the sdpId, then COUNT(1) per (sdpId, status).
DISTINCT_STATUSES_QUERY = """
SELECT DISTINCT VALUE c.status
FROM c
WHERE c.sdpId = @sdpId
"""

COUNT_BY_STATUS_QUERY = """
SELECT VALUE COUNT(1)
FROM c
WHERE c.sdpId = @sdpId AND c.status = @status
"""

rows = []
for sdp_id in SDP_IDS:
    statuses = list(container.query_items(
        query=DISTINCT_STATUSES_QUERY,
        parameters=[{"name": "@sdpId", "value": sdp_id}],
        enable_cross_partition_query=True,
    ))
    if not statuses:
        rows.append({"sdpId": sdp_id, "status": None, "docCount": 0})
        continue
    for status in statuses:
        count_res = list(container.query_items(
            query=COUNT_BY_STATUS_QUERY,
            parameters=[
                {"name": "@sdpId",  "value": sdp_id},
                {"name": "@status", "value": status},
            ],
            enable_cross_partition_query=True,
        ))
        rows.append({
            "sdpId":    sdp_id,
            "status":   status,
            "docCount": count_res[0] if count_res else 0,
        })

stats_df = pd.DataFrame(rows)

# Pivot: one row per sdpId, one column per status
pivot_df = (
    stats_df
    .pivot_table(index="sdpId", columns="status", values="docCount", aggfunc="sum", fill_value=0)
    .reset_index()
)
pivot_df["total"] = pivot_df.drop(columns=["sdpId"]).sum(axis=1)

print(f"Queried {len(SDP_IDS)} sdpIds, got {len(stats_df)} (sdpId, status) rows")

# Show the full sdpId without pandas truncating long strings
with pd.option_context("display.max_colwidth", None, "display.width", None):
    display(pivot_df)

## 5. FAILED messages grouped by `errorType`

For each sdpId, count `status = FAILED` documents grouped by `errorDetails.errorType`. Uses the same `DISTINCT VALUE` + `COUNT(1)` two-query workaround because the Cosmos client does not support non-`VALUE` `GROUP BY` aggregates. If your schema stores the error type under a different path (e.g. `c.error.type` or `c.failureReason`), adjust the field reference in both queries.

In [0]:
import pandas as pd

DISTINCT_ERROR_TYPES_QUERY = """
SELECT DISTINCT VALUE c.errorDetails.errorType
FROM c
WHERE c.sdpId = @sdpId AND c.status = 'FAILED'
"""

COUNT_BY_ERROR_TYPE_QUERY = """
SELECT VALUE COUNT(1)
FROM c
WHERE c.sdpId = @sdpId AND c.status = 'FAILED' AND c.errorDetails.errorType = @errorType
"""

error_rows = []
for sdp_id in SDP_IDS:
    error_types = list(container.query_items(
        query=DISTINCT_ERROR_TYPES_QUERY,
        parameters=[{"name": "@sdpId", "value": sdp_id}],
        enable_cross_partition_query=True,
    ))
    if not error_types:
        continue
    for error_type in error_types:
        count_res = list(container.query_items(
            query=COUNT_BY_ERROR_TYPE_QUERY,
            parameters=[
                {"name": "@sdpId",     "value": sdp_id},
                {"name": "@errorType", "value": error_type},
            ],
            enable_cross_partition_query=True,
        ))
        error_rows.append({
            "sdpId":     sdp_id,
            "errorType": error_type if error_type is not None else "<missing>",
            "docCount":  count_res[0] if count_res else 0,
        })

error_stats_df = pd.DataFrame(error_rows)

if error_stats_df.empty:
    print("No FAILED documents found across the given sdpIds (or errorType field path is different).")
else:
    error_pivot_df = (
        error_stats_df
        .pivot_table(index="sdpId", columns="errorType", values="docCount", aggfunc="sum", fill_value=0)
        .reset_index()
    )
    error_pivot_df["total_failed"] = error_pivot_df.drop(columns=["sdpId"]).sum(axis=1)

    print(f"Queried {len(SDP_IDS)} sdpIds, found {len(error_stats_df)} (sdpId, errorType) rows")
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(error_pivot_df.sort_values("total_failed", ascending=False))

## 6. Error details analysis

Drills into actual FAILED documents to understand the failures. Pulls a sample of FAILED docs per sdpId with the key error fields (`errorType`, `errorMessage`, `stage`, `sourceAptId`, `_ts`), then aggregates the top errorType / errorMessage combinations across all sdpIds.

In [0]:
import pandas as pd

SAMPLE_PER_SDP = 10  # max FAILED docs to pull per sdpId for inspection

FAILED_SAMPLE_QUERY = f"""
SELECT TOP {SAMPLE_PER_SDP}
    c.sdpId,
    c.id,
    c.stage,
    c.errorDetails.errorType    AS errorType,
    c.errorDetails.errorMessage AS errorMessage,
    c.errorDetails.stage        AS errorStage,
    c.errorDetails.details      AS errorDetailsField,
    c.rawSourceApt.sourceAptId  AS sourceAptId,
    c.rawSourceApt.countryIso   AS countryIso,
    c._ts                       AS ts
FROM c
WHERE c.sdpId = @sdpId AND c.status = 'FAILED'
"""

sample_rows = []
for sdp_id in SDP_IDS:
    docs = list(container.query_items(
        query=FAILED_SAMPLE_QUERY,
        parameters=[{"name": "@sdpId", "value": sdp_id}],
        enable_cross_partition_query=True,
    ))
    sample_rows.extend(docs)

failed_sample_df = pd.DataFrame(sample_rows)

for col in ["sdpId", "id", "stage", "errorType", "errorMessage",
            "errorStage", "errorDetailsField", "sourceAptId", "countryIso", "ts"]:
    if col not in failed_sample_df.columns:
        failed_sample_df[col] = None

if failed_sample_df.empty:
    print("No FAILED sample documents found.")
else:
    failed_sample_df["timestamp"] = pd.to_datetime(failed_sample_df["ts"], unit="s", utc=True, errors="coerce")
    populated = {c: failed_sample_df[c].notna().sum() for c in
                 ["errorType", "errorMessage", "errorStage", "errorDetailsField", "stage"]}
    print(f"Pulled {len(failed_sample_df)} FAILED sample documents (up to {SAMPLE_PER_SDP} per sdpId).")
    print(f"Field population (non-null counts): {populated}")
    print()

    print("Failures by errorType:")
    by_error_type = (
        failed_sample_df
        .groupby("errorType", dropna=False)
        .size()
        .reset_index(name="sampleCount")
        .sort_values("sampleCount", ascending=False)
    )
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(by_error_type)

    print("Failures by stage (the pipeline stage where the message failed):")
    by_stage = (
        failed_sample_df
        .groupby("stage", dropna=False)
        .size()
        .reset_index(name="sampleCount")
        .sort_values("sampleCount", ascending=False)
    )
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(by_stage)

    print("errorType x stage cross-tab:")
    crosstab = pd.crosstab(
        failed_sample_df["errorType"].fillna("<missing>"),
        failed_sample_df["stage"].fillna("<missing>"),
    )
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(crosstab)

    print("errorType + errorMessage combinations:")
    by_error_combo = (
        failed_sample_df
        .groupby(["errorType", "errorMessage"], dropna=False)
        .size()
        .reset_index(name="sampleCount")
        .sort_values("sampleCount", ascending=False)
    )
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(by_error_combo)

    print("Raw FAILED sample (first 20 rows):")
    with pd.option_context("display.max_colwidth", None, "display.width", None):
        display(failed_sample_df[["errorType", "errorMessage", "stage", "errorStage",
                                  "errorDetailsField", "sourceAptId", "countryIso",
                                  "timestamp"]].head(20))